In [178]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder, StandardScaler
import time





### Openml
In Python, OpenML is mainly used to discover, download, and share ML datasets, tasks, and results—super handy for experiments, benchmarking, and learning ML properly.

In [135]:
%pip install openml



Note: you may need to restart the kernel to use updated packages.


In [136]:
import openml

In [137]:
# check the datasets with ≤ 10,000 rows i
df = openml.datasets.list_datasets(
    output_format="dataframe",
    number_instances=100
)

df_multiclass = df[df["NumberOfClasses"] >= 2]

df_multiclass


,did,name,version,uploader,status,format,MajorityClassSize,MaxNominalAttDistinctValues,MinorityClassSize,NumberOfClasses,NumberOfFeatures,NumberOfInstances,NumberOfInstancesWithMissingValues,NumberOfMissingValues,NumberOfNumericFeatures,NumberOfSymbolicFeatures
461,461,analcatdata_creditscore,1,2,active,ARFF,73.0,6.0,27.0,2.0,7.0,100.0,0.0,0.0,3.0,4.0
716,716,fri_c3_100_50,2,2,active,ARFF,62.0,2.0,38.0,2.0,51.0,100.0,0.0,0.0,50.0,1.0
726,726,fri_c2_100_5,2,2,active,ARFF,60.0,2.0,40.0,2.0,6.0,100.0,0.0,0.0,5.0,1.0
754,754,fri_c0_100_5,2,2,active,ARFF,54.0,2.0,46.0,2.0,6.0,100.0,0.0,0.0,5.0,1.0
762,762,fri_c2_100_10,2,2,active,ARFF,55.0,2.0,45.0,2.0,11.0,100.0,0.0,0.0,10.0,1.0
768,768,fri_c3_100_25,2,2,active,ARFF,55.0,2.0,45.0,2.0,26.0,100.0,0.0,0.0,25.0,1.0
775,775,fri_c2_100_25,2,2,active,ARFF,57.0,2.0,43.0,2.0,26.0,100.0,0.0,0.0,25.0,1.0
783,783,fri_c3_100_10,2,2,active,ARFF,60.0,2.0,40.0,2.0,11.0,100.0,0.0,0.0,10.0,1.0
789,789,fri_c1_100_10,2,2,active,ARFF,53.0,2.0,47.0,2.0,11.0,100.0,0.0,0.0,10.0,1.0
808,808,fri_c0_100_10,2,2,active,ARFF,55.0,2.0,45.0,2.0,11.0,100.0,0.0,0.0,10.0,1.0


In [138]:
df = openml.datasets.list_datasets(output_format="dataframe")

df[df["name"].str.lower() == "iris"]

,did,name,version,uploader,status,format,MajorityClassSize,MaxNominalAttDistinctValues,MinorityClassSize,NumberOfClasses,NumberOfFeatures,NumberOfInstances,NumberOfInstancesWithMissingValues,NumberOfMissingValues,NumberOfNumericFeatures,NumberOfSymbolicFeatures
61,61,iris,1,1,active,ARFF,50.0,3.0,50.0,3.0,5.0,150.0,0.0,0.0,4.0,1.0
969,969,iris,3,2,active,ARFF,100.0,2.0,50.0,2.0,5.0,150.0,0.0,0.0,4.0,1.0
41510,41510,iris,9,348,active,ARFF,NaN,3.0,NaN,NaN,5.0,150.0,0.0,0.0,4.0,1.0
41511,41511,iris,10,348,active,ARFF,50.0,3.0,50.0,3.0,5.0,150.0,0.0,0.0,4.0,1.0
41567,41567,iris,11,348,active,ARFF,NaN,3.0,NaN,NaN,5.0,150.0,0.0,0.0,4.0,1.0
41568,41568,iris,12,348,active,ARFF,50.0,3.0,50.0,3.0,5.0,150.0,0.0,0.0,4.0,1.0
41582,41582,iris,13,348,active,ARFF,NaN,3.0,NaN,NaN,5.0,150.0,0.0,0.0,4.0,1.0
41583,41583,iris,14,348,active,ARFF,50.0,3.0,50.0,3.0,5.0,150.0,0.0,0.0,4.0,1.0
41996,41996,iris,15,348,active,ARFF,NaN,3.0,NaN,NaN,5.0,150.0,0.0,0.0,4.0,1.0
41997,41997,iris,16,348,active,ARFF,50.0,3.0,50.0,3.0,5.0,150.0,0.0,0.0,4.0,1.0


In [139]:
# Download metadata using openml
dataset = openml.datasets.get_dataset(61)

In [140]:
dataset

OpenML Dataset
Name.........: iris
Version......: 1
Format.......: ARFF
Upload Date..: 2014-04-06 23:23:39
Licence......: Public
Download URL.: https://openml.org/data/v1/download/61/iris.arff
OpenML URL...: https://www.openml.org/d/61
# of features: None

In [141]:
# Download the actual data
X,y, categorical_indicator, attribute_names = dataset.get_data(
    target=dataset.default_target_attribute
)

# Save dataset locally
df = X.copy()


In [142]:
df.head()

,sepallength,sepalwidth,petallength,petalwidth
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [143]:
df.shape

(150, 4)

In [144]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sepallength  150 non-null    float64
 1   sepalwidth   150 non-null    float64
 2   petallength  150 non-null    float64
 3   petalwidth   150 non-null    float64
dtypes: float64(4)
memory usage: 4.8 KB


### Prepare data

In [145]:
columns = df.columns
df.dtypes

sepallength    float64
sepalwidth     float64
petallength    float64
petalwidth     float64
dtype: object

### Manually select the columns type 

In [146]:
# Define columns types
num_columns = columns
#ordinal_columns = ["education", "risk"]
#ordinal_order = ["primary", "secondary", "tertiary"],     # education
#    ["low", "medium", "high", "very_high"]    # risk
#nominal_columns = []
# bin_cols = ["default", "housing", "loan", "y"]


# Pre-processing
Numeric variables --> scale
Ordinal variables -->  Ordinal encoding
Nominal categorical variables -->  Onehot encoding
Binary(already 0 and 1) -->  onehot encoding 


# Preprocess

In [147]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder, MinMaxScaler
from sklearn.cluster import KMeans


In [148]:
preprocessor = ColumnTransformer(
    transformers = [
        ("num", MinMaxScaler(), num_columns),
        #("nominal", OneHotEncoder(handle_unknown="ignore"), nominal_columns),
        #("ordinal", OrdinalEncoder(categories=ordinal_order), ordinal_columns),
        #("bin", "passthrough", bin_cols) # data is already numeric and no need totransform this column
    ]
)

### Joblib 
joblib is a small but powerful Python library mainly used in ML for saving models, fast loading, and parallel processing

n_jobs answers “how many things can run in parallel? n_jobs does NOT say threads or processes.
Threading is how parallelism is done. Threading is a backend choice in joblib.
backend="threading"   # threads
backend="loky"        # processes (default)

In [149]:
%pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [150]:
from joblib import Parallel, delayed
import itertools # to create combinations of hyperparameters

# Define parameters

In [151]:
param_grid = {
    "n_clusters": [2, 3, 4, 5, 6],
    "init": ["k-means++", "random"],
    "n_init": [10, 20],
    "max_iter": [300, 500]
}

param_combinations =list(itertools.product(
    param_grid["n_clusters"],
    param_grid["init"],
    param_grid["n_init"],
    param_grid["max_iter"]
))

# Define model

In [ ]:
def evaluate_kmeans(params):
    start = time.time()
    n_clusters, init, n_init, max_iter = params
    
    pipe = Pipeline([
        ("scale", StandardScaler()),
        ("kmeans", KMeans(
            n_clusters=n_clusters,
            init=init,
            n_init=n_init,
            max_iter=max_iter,
            random_state=42
        ))
    ])
    
    labels = pipe.fit_predict(df)

    # silhouette_score
    silhouette = silhouette_score(df, labels)

    # Supervised clustering metrics
    ari = adjusted_rand_score(y, labels) # no need to do label alignment

    execution_time = float(time.time() - start)

    return {
       "n_clusters": n_clusters,
        "init": init,
        "n_init": n_init,
        "max_iter": max_iter,
        "silhouette": silhouette,
        "inertia": pipe.named_steps["kmeans"].inertia_,
        "execution_time": execution_time,
        "adjusted_rand_score": ari
        
    }


results = Parallel(n_jobs=-1, verbose=10)(
    delayed(evaluate_kmeans)(params) for params in param_combinations
    )
results_kmeans =pd.DataFrame(results)
print(results_kmeans)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    7.3s
[Parallel(n_jobs=-1)]: Done  14 out of  40 | elapsed:    7.3s remaining:   13.7s
[Parallel(n_jobs=-1)]: Done  19 out of  40 | elapsed:    7.4s remaining:    8.2s
[Parallel(n_jobs=-1)]: Done  24 out of  40 | elapsed:    7.4s remaining:    4.9s
[Parallel(n_jobs=-1)]: Done  29 out of  40 | elapsed:    7.4s remaining:    2.8s
[Parallel(n_jobs=-1)]: Done  34 out of  40 | elapsed:    7.5s remaining:    1.2s
[Parallel(n_jobs=-1)]: Done  40 out of  40 | elapsed:    7.6s finished


    n_clusters       init  n_init  max_iter  silhouette     inertia  \
0            2  k-means++      10       300    0.686393  223.732006   
1            2  k-means++      10       500    0.686393  223.732006   
2            2  k-means++      20       300    0.686393  223.732006   
3            2  k-means++      20       500    0.686393  223.732006   
4            2     random      10       300    0.686393  223.732006   
5            2     random      10       500    0.686393  223.732006   
6            2     random      20       300    0.686393  223.732006   
7            2     random      20       500    0.686393  223.732006   
8            3  k-means++      10       300    0.505931  140.965817   
9            3  k-means++      10       500    0.505931  140.965817   
10           3  k-means++      20       300    0.505931  140.965817   
11           3  k-means++      20       500    0.505931  140.965817   
12           3     random      10       300    0.503001  140.968379   
13    